# 07 置信度阈值分析

**用途**：遍历多个阈值，统计各阈值下的检测数量，绘制折线图，辅助选择最优工作阈值。


In [5]:
# ── 0. 配置区 ──────────────────────────────────────────────
import numpy as np

DATASET_NAME = "sahi_null_v2_ms1_0605-0621_40_ok"
PRED_FIELD   = "small_slices_a03_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_16"
GT_FIELD     = "ground_truth"
THR_RANGE    = list(np.arange(0.1, 0.96, 0.05).round(2))  # [0.1, 0.15, ..., 0.95]
TARGET_THR   = 0.5   # 最终查看的目标阈值


In [2]:
# ── 1. 初始化日志 ──────────────────────────────────────────
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ 07_confidence_threshold_analysis 初始化完成 ============")


2026-02-19 16:19:37,525 [INFO] ============ 07_confidence_threshold_analysis 初始化完成 ============


## Step 1: 加载数据集


In [3]:
import fiftyone as fo
import pandas as pd
from IPython.display import display
from fiftyone import ViewField as F

logger.info("Step 1 开始：加载数据集")

ds = fo.load_dataset(DATASET_NAME)
print(f"数据集: {ds.name}，样本数: {len(ds)}")

# 统计 GT 基准数量
if GT_FIELD in ds.get_field_schema():
    gt_dets = ds.values(f"{GT_FIELD}.detections")
    gt_total = sum(len(d) for d in gt_dets if d)
    print(f"GT 标注总数（{GT_FIELD}）: {gt_total}")
else:
    gt_total = None
    logger.warning(f"GT_FIELD='{GT_FIELD}' 不存在，不绘制 GT 基准线")

logger.info(f"Step 1 完成：gt_total={gt_total}")


2026-02-19 16:19:44,123 [INFO] Step 1 开始：加载数据集
2026-02-19 16:19:44,306 [INFO] Step 1 完成：gt_total=850


数据集: sahi_null_v2_ms1_0605-0621_40_ok，样本数: 243
GT 标注总数（ground_truth）: 850


## Step 2: 按阈值统计检测数


In [6]:
logger.info("Step 2 开始：按阈值遍历")

rows = []
for thr in THR_RANGE:
    view = ds.filter_labels(PRED_FIELD, F("confidence") > thr, only_matches=False)
    pred_dets = view.values(f"{PRED_FIELD}.detections")
    pred_total = sum(len(d) for d in pred_dets if d)
    rows.append({"threshold": thr, "pred_count": pred_total})
    logger.info(f"  thr={thr:.2f} -> pred_count={pred_total}")

thr_df = pd.DataFrame(rows)
display(thr_df)

logger.info(f"Step 2 完成：共统计 {len(THR_RANGE)} 个阈值")


2026-02-19 16:20:51,077 [INFO] Step 2 开始：按阈值遍历
2026-02-19 16:20:51,164 [INFO]   thr=0.10 -> pred_count=2454
2026-02-19 16:20:51,310 [INFO]   thr=0.15 -> pred_count=2307
2026-02-19 16:20:51,388 [INFO]   thr=0.20 -> pred_count=2200
2026-02-19 16:20:51,462 [INFO]   thr=0.25 -> pred_count=2116
2026-02-19 16:20:51,533 [INFO]   thr=0.30 -> pred_count=2053
2026-02-19 16:20:51,669 [INFO]   thr=0.35 -> pred_count=1990
2026-02-19 16:20:51,737 [INFO]   thr=0.40 -> pred_count=1933
2026-02-19 16:20:51,802 [INFO]   thr=0.45 -> pred_count=1876
2026-02-19 16:20:51,932 [INFO]   thr=0.50 -> pred_count=1825
2026-02-19 16:20:51,994 [INFO]   thr=0.55 -> pred_count=1762
2026-02-19 16:20:52,053 [INFO]   thr=0.60 -> pred_count=1709
2026-02-19 16:20:52,111 [INFO]   thr=0.65 -> pred_count=1645
2026-02-19 16:20:52,234 [INFO]   thr=0.70 -> pred_count=1573
2026-02-19 16:20:52,286 [INFO]   thr=0.75 -> pred_count=1503
2026-02-19 16:20:52,336 [INFO]   thr=0.80 -> pred_count=1396
2026-02-19 16:20:52,382 [INFO]   thr=0

,threshold,pred_count
0,0.10,2454
1,0.15,2307
2,0.20,2200
3,0.25,2116
4,0.30,2053
5,0.35,1990
6,0.40,1933
7,0.45,1876
8,0.50,1825
9,0.55,1762


2026-02-19 16:20:52,425 [INFO] Step 2 完成：共统计 18 个阈值


## Step 3: 绘制曲线


In [9]:
import plotly.graph_objects as go

logger.info("Step 3 开始：绘图")

fig = go.Figure()

# ===== 主曲线 =====
fig.add_trace(
    go.Scatter(
        x=thr_df["threshold"],
        y=thr_df["pred_count"],
        mode="lines+markers",
        name="预测数量",
    )
)

# ===== GT baseline =====
if gt_total is not None:
    fig.add_hline(
        y=gt_total,
        line=dict(color="red", dash="dash"),
        annotation_text=f"GT 基准 ({gt_total})",
        annotation_position="top left",
    )

# ===== Target threshold =====
fig.add_vline(
    x=TARGET_THR,
    line=dict(color="green", dash="dot"),
    annotation_text=f"目标阈值 ({TARGET_THR})",
    annotation_position="top right",
)

# ===== Layout =====
fig.update_layout(
    title=f"阈值 vs 检测数量<br>{DATASET_NAME} / {PRED_FIELD}",
    xaxis_title="置信度阈值",
    yaxis_title="检测框数量",
    template="plotly_white",
    width=1000,
    height=500,
)

fig.show()

logger.info("Step 3 完成：图表已绘制")


2026-02-19 16:23:56,674 [INFO] Step 3 开始：绘图


2026-02-19 16:23:56,771 [INFO] Step 3 完成：图表已绘制
